# Assignment 4 — Optimizing Transformer Translation with Ray Tune & Optuna
### English → Hindi Neural Machine Translation | Hyperparameter Optimization


In [1]:
import os

if os.path.exists('/kaggle/working'):
    SAVE_DIR  = '/kaggle/working'
    DATA_PATH = '/kaggle/input/datasets/akankshakapil/mydataset/English-Hindi.tsv'


In [2]:
!pip install -q ray optuna
from ray import tune
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import ASHAScheduler


In [3]:
# Install Ray Tune and Optuna
import subprocess, sys

print("Installing ray[tune]...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ray[tune]", "optuna"],
               check=True)

print("\nInstalling NLTK data...")
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("\n✅ All dependencies installed!")

Installing ray[tune]...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 2.8 MB/s eta 0:00:00

Installing NLTK data...

✅ All dependencies installed!


In [4]:
import os, math, time, pickle, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter

import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

import ray
from ray import tune
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import ASHAScheduler

# ── Device ────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device   : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Global constants ──────────────────────────────────────────────────────
MAX_LEN        = 50
FREQ_THRESHOLD = 2
D_MODEL        = 512   

Device   : cuda
GPU      : Tesla T4
VRAM     : 15.6 GB


## Vocabulary Class

In [5]:
class Vocabulary:
    """
    Bidirectional word ↔ index mapping.
    Special tokens: <pad>=0, <sos>=1, <eos>=2, <unk>=3
    Words appearing < freq_threshold times are mapped to <unk>.
    """
    def __init__(self, freq_threshold=2):
        self.freq_threshold = freq_threshold
        self.itos = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.stoi = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.idx  = 4

    def build_vocab(self, sentence_list):
        freq = Counter()
        for s in sentence_list:
            for w in self.tokenize(s):
                freq[w] += 1
        for word, f in freq.items():
            if f >= self.freq_threshold:
                self.stoi[word]    = self.idx
                self.itos[self.idx] = word
                self.idx += 1

    def tokenize(self, sentence):
        return sentence.lower().strip().split()

    def numericalize(self, sentence):
        return [self.stoi.get(w, self.stoi["<unk>"]) for w in self.tokenize(sentence)]

    def __len__(self):   return len(self.stoi)
    def __getitem__(self, t): return self.stoi.get(t, self.stoi["<unk>"])

print("Vocabulary class defined ✓")

Vocabulary class defined ✓


## Data Loading & Preprocessing

In [6]:
def encode_sentence(sentence, vocab, max_len=MAX_LEN):
    """Convert sentence → fixed-length padded integer list."""
    tokens = ([vocab.stoi["<sos>"]]
              + vocab.numericalize(sentence)[:max_len - 2]
              + [vocab.stoi["<eos>"]])
    return tokens + [vocab.stoi["<pad>"]] * (max_len - len(tokens))


class TranslationDataset(Dataset):
    def __init__(self, df, en_vocab, hi_vocab, max_len=MAX_LEN):
        self.en       = df["en"].tolist()
        self.hi       = df["hi"].tolist()
        self.en_vocab = en_vocab
        self.hi_vocab = hi_vocab
        self.max_len  = max_len

    def __len__(self): return len(self.en)

    def __getitem__(self, idx):
        src = encode_sentence(self.en[idx], self.en_vocab, self.max_len)
        tgt = encode_sentence(self.hi[idx], self.hi_vocab, self.max_len)
        return torch.tensor(src), torch.tensor(tgt)


def collate_fn(batch):
    """
    Teacher forcing split:
      tgt_input  = <sos> word1 word2 ...        (decoder input)
      tgt_output = word1 word2 ... <eos>        (what we predict)
    """
    srcs, tgts = zip(*batch)
    srcs = torch.stack(srcs)
    tgts = torch.stack(tgts)
    return srcs, tgts[:, :-1], tgts[:, 1:]


# ── Load & preprocess ─────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH, sep="\t", header=None, names=["id1","en","id2","hi"])
df = df[["en","hi"]].dropna().reset_index(drop=True)

en_vocab = Vocabulary(FREQ_THRESHOLD)
hi_vocab = Vocabulary(FREQ_THRESHOLD)
en_vocab.build_vocab(df["en"].tolist())
hi_vocab.build_vocab(df["hi"].tolist())

print(f"Total sentence pairs : {len(df):,}")
print(f"English vocabulary   : {len(en_vocab):,} words")
print(f"Hindi vocabulary     : {len(hi_vocab):,} words")
df.head(3)

Total sentence pairs : 13,186
English vocabulary   : 4,117 words
Hindi vocabulary     : 4,044 words


,en,hi
0,Muiriel is 20 now.,म्यूरियल अब बीस साल की हो गई है।
1,Muiriel is 20 now.,म्यूरियल अब बीस साल की है।
2,Education in this world disappoints me.,मैं इस दुनिया में शिक्षा पर बहुत निराश हूँ।


## Transformer Model Architecture

In [7]:
# ── Positional Encoding ───────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    """Sine/cosine positional encoding (Vaswani et al., 2017)."""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe       = torch.zeros(max_len, d_model)
        pos      = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float()
                             * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


# ── Multi-Head Attention ───────────────────────────────────────────────────
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k       = d_model // num_heads
        self.num_heads = num_heads
        self.d_model   = d_model
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        def proj(W, x):
            return W(x).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        Q, K, V = proj(self.W_q, q), proj(self.W_k, k), proj(self.W_v, v)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        w   = self.drop(torch.softmax(scores, dim=-1))
        out = torch.matmul(w, V).transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.W_o(out)


# ── Feed-Forward ──────────────────────────────────────────────────────────
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model))
    def forward(self, x): return self.net(x)


# ── Layer Norm ────────────────────────────────────────────────────────────
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta  = nn.Parameter(torch.zeros(d_model))
        self.eps   = eps
    def forward(self, x):
        m = x.mean(-1, keepdim=True)
        s = x.std(-1,  keepdim=True)
        return self.gamma * (x - m) / (s + self.eps) + self.beta


# ── Encoder / Decoder Layers ──────────────────────────────────────────────
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn   = FeedForward(d_model, d_ff, dropout)
        self.norm1 = LayerNorm(d_model); self.norm2 = LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)
    def forward(self, x, mask=None):
        x = self.norm1(x + self.drop(self.attn(x, x, x, mask)))
        return self.norm2(x + self.drop(self.ffn(x)))


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.s_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.c_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn    = FeedForward(d_model, d_ff, dropout)
        self.norm1  = LayerNorm(d_model); self.norm2 = LayerNorm(d_model)
        self.norm3  = LayerNorm(d_model); self.drop  = nn.Dropout(dropout)
    def forward(self, x, enc, src_mask=None, tgt_mask=None):
        x = self.norm1(x + self.drop(self.s_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.drop(self.c_attn(x, enc, enc, src_mask)))
        return self.norm3(x + self.drop(self.ffn(x)))


# ── Full Transformer ──────────────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, vocab, d, nl, nh, dff, ml, dr):
        super().__init__()
        self.embed  = nn.Embedding(vocab, d)
        self.pos    = PositionalEncoding(d, ml)
        self.layers = nn.ModuleList([EncoderLayer(d, nh, dff, dr) for _ in range(nl)])
        self.drop   = nn.Dropout(dr)
    def forward(self, x, mask=None):
        x = self.drop(self.pos(self.embed(x)))
        for l in self.layers: x = l(x, mask)
        return x


class Decoder(nn.Module):
    def __init__(self, vocab, d, nl, nh, dff, ml, dr):
        super().__init__()
        self.embed  = nn.Embedding(vocab, d)
        self.pos    = PositionalEncoding(d, ml)
        self.layers = nn.ModuleList([DecoderLayer(d, nh, dff, dr) for _ in range(nl)])
        self.drop   = nn.Dropout(dr)
    def forward(self, x, enc, sm=None, tm=None):
        x = self.drop(self.pos(self.embed(x)))
        for l in self.layers: x = l(x, enc, sm, tm)
        return x


class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab,
                 d_model=512, num_layers=6, num_heads=8,
                 d_ff=2048, max_len=100, dropout=0.1):
        super().__init__()
        self.enc    = Encoder(src_vocab, d_model, num_layers, num_heads, d_ff, max_len, dropout)
        self.dec    = Decoder(tgt_vocab, d_model, num_layers, num_heads, d_ff, max_len, dropout)
        self.fc_out = nn.Linear(d_model, tgt_vocab)

    def _pad_mask(self, seq, pad):
        return (seq != pad).unsqueeze(1).unsqueeze(2)

    def _causal_mask(self, sz):
        return torch.tril(torch.ones(sz, sz)).bool().to(next(self.parameters()).device)

    def forward(self, src, tgt, sp, tp):
        sm = self._pad_mask(src, sp)
        tm = self._pad_mask(tgt, tp) & self._causal_mask(tgt.size(1))
        e  = self.enc(src, sm)
        d  = self.dec(tgt, e, sm, tm)
        return self.fc_out(d)


print("Transformer model defined ✓")

Transformer model defined ✓


---
## Part 1 — Baseline Metrics

In [8]:
# ── Baseline metrics from original en_to_hi.ipynb ────────────────────────
# Fill in final_loss and training_time_min after running original notebook.
BASELINE = {
    "epochs"            : 100,
    "bleu_score"        : 0.5247,      
    "bleu_pct"          : 52.47,
    "final_loss"        : 0.0972,        
    "training_time_min" : 128,        
    "config": {
        "lr": 1e-4, "batch_size": 60, "d_model": 512,
        "num_layers": 6, "num_heads": 8, "d_ff": 2048, "dropout": 0.1
    }
}

print("=" * 55)
print("PART 1 — BASELINE METRICS")
print(f"  Epochs              : {BASELINE['epochs']}")
print(f"  BLEU Score          : {BASELINE['bleu_pct']:.2f}%")
print(f"  Final Loss          : {BASELINE['final_loss']}")
print(f"  Training Time (min) : {BASELINE['training_time_min']}")
print("=" * 55)

PART 1 — BASELINE METRICS
  Epochs              : 100
  BLEU Score          : 52.47%
  Final Loss          : 0.0972
  Training Time (min) : 128


---
## Part 2 — Ray Tune Training Function

In [9]:
def train_tune(config):
    """
    Ray Tune compatible training function.
    Accepts a hyperparameter config dict, trains the Transformer,
    and reports loss to Ray Tune after every epoch so ASHA can
    prune underperforming trials early.

    Research-backed choices inside:
    - Adam betas=(0.9, 0.98), eps=1e-9  → Vaswani et al. (2017)
    - Gradient clipping max_norm=1.0    → standard for Transformers
    - Label smoothing 0.1               → Szegedy et al. (2016)
    - CosineAnnealingLR                 → smooth LR decay, no sharp drops
    """
    import os, math, pandas as pd
    import torch, torch.nn as nn, torch.optim as optim
    from torch.utils.data import Dataset, DataLoader
    from collections import Counter
    from ray import tune as _tune

    # ── Device ───────────────────────────────────────────────────────────
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── Data (rebuilt per trial — fast, < 5s) ────────────────────────────
    _df = pd.read_csv(config["data_path"], sep="\t", header=None,
                      names=["id1","en","id2","hi"])
    _df = _df[["en","hi"]].dropna().reset_index(drop=True)

    _en_vocab = Vocabulary(freq_threshold=2)
    _hi_vocab = Vocabulary(freq_threshold=2)
    _en_vocab.build_vocab(_df["en"].tolist())
    _hi_vocab.build_vocab(_df["hi"].tolist())

    src_pad = _en_vocab["<pad>"]
    tgt_pad = _hi_vocab["<pad>"]

    _dataset = TranslationDataset(_df, _en_vocab, _hi_vocab, max_len=MAX_LEN)
    _loader  = DataLoader(
        _dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=2,        
        pin_memory=True,      
    )

    # ── Model ────────────────────────────────────────────────────────────
    model = Transformer(
        src_vocab  = len(_en_vocab),
        tgt_vocab  = len(_hi_vocab),
        d_model    = D_MODEL,
        num_layers = config["num_layers"],
        num_heads  = config["num_heads"],
        d_ff       = config["d_ff"],
        max_len    = MAX_LEN,
        dropout    = config["dropout"],
    ).to(device)

    # ── Optimizer + LR Scheduler ─────────────────────────────────────────
    optimizer = optim.Adam(
        model.parameters(),
        lr=config["lr"],
        betas=(0.9, 0.98),
        eps=1e-9,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max   = config["num_epochs"],
        eta_min = config["lr"] * 0.1,
    )

    # label_smoothing reduces overconfidence — consistently helps NMT quality
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_pad, label_smoothing=0.1)

    # ── Training Loop ────────────────────────────────────────────────────
    for epoch in range(config["num_epochs"]):
        model.train()
        epoch_loss = 0.0

        for src, tgt_in, tgt_out in _loader:
            src, tgt_in, tgt_out = (src.to(device), tgt_in.to(device),
                                    tgt_out.to(device))
            logits = model(src, tgt_in, src_pad, tgt_pad)
            loss   = criterion(logits.view(-1, logits.shape[-1]),
                               tgt_out.view(-1))

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(_loader)

        # ← This line is what ASHA uses to prune bad trials
        _tune.report({"loss": avg_loss, "epoch": epoch + 1})


print("train_tune() defined ✓")

train_tune() defined ✓


## Part 2.2 — Search Space (6 Hyperparameters)

| # | Hyperparameter | Range | Rationale |
|---|---|---|---|
| 1 | Learning Rate | 1e-5 → 1e-3 (log) | Log scale is standard for lr search (Bergstra & Bengio, 2012) |
| 2 | Batch Size | 32, 64, 128 | Powers of 2 align with GPU memory; original used non-standard 60 |
| 3 | Attention Heads | 4, 8 | Both divide D_MODEL=512; 4 heads is ~2× faster |
| 4 | FeedForward Dim | 1024, 2048, 4096 | Controls model capacity; original used 2048 |
| 5 | Dropout | 0.05 → 0.35 | Wider range to find better regularization for 13k-pair dataset |
| 6 | Num Layers | 3, 4, 6 | Fewer layers often converge faster on small datasets |

In [10]:
from ray import tune

SEARCH_SPACE = {
    "data_path"  : DATA_PATH,   
    "num_epochs" : 30,          

    # ── 6 Hyperparameters to tune ─────────────────────────────────────────
    "lr"         : tune.loguniform(1e-5, 1e-3),     # HP 1: log-uniform learning rate
    "batch_size" : tune.choice([16 , 32, 64]),      # HP 2: batch size
    "num_heads"  : tune.choice([4, 8]),             # HP 3: attention heads (must divide 512)
    "d_ff"       : tune.choice([1024, 2048]),       # HP 4: feedforward hidden dim
    "dropout"    : tune.uniform(0.05, 0.35),        # HP 5: dropout rate
    "num_layers" : tune.choice([3, 4]),          # HP 6: encoder/decoder depth
}

print("Search space defined ✓")
print("Hyperparameters to tune:", [k for k in SEARCH_SPACE if k not in ("data_path","num_epochs")])

Search space defined ✓
Hyperparameters to tune: ['lr', 'batch_size', 'num_heads', 'd_ff', 'dropout', 'num_layers']


---
## Part 2.3 & 3 — Run Ray Tune Sweep

In [12]:
import ray
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import ASHAScheduler

# Shut down any previous Ray session cleanly
if ray.is_initialized():
    ray.shutdown()

ray.init(
    ignore_reinit_error=True,
    # num_cpus=2,
    num_gpus=1,                    # Tell Ray we have 1 GPU
    include_dashboard=False,       # Dashboard doesn't work in Colab
    log_to_driver=False,           # Reduces verbose output
    _temp_dir="/tmp/ray",          # Stable temp dir for Colab environment
)

# ── Optuna TPE Search ─────────────────────────────────────────────────────.
optuna_search = OptunaSearch(metric="loss", mode="min")

# ── ASHA Scheduler ────────────────────────────────────────────────────────
asha = ASHAScheduler(
    metric           = "loss",
    mode             = "min",
    max_t            = 30,   # Max epochs per trial
    grace_period     = 5,    # Min epochs before a trial can be killed
    reduction_factor = 2,    # Keep top 50% at each rung
    brackets         = 1,
)

# ── Run Tuner ─────────────────────────────────────────────────────────────
tuner = tune.Tuner(
    tune.with_resources(
        train_tune,
        resources={ "gpu": 1},  # 1 GPU per trial
    ),
    tune_config=tune.TuneConfig(
        search_alg          = optuna_search,
        scheduler           = asha,
        num_samples         = 12,   # 15 different hyperparameter combinations
        max_concurrent_trials = 1,  # 1 at a time (single GPU)
    ),
    param_space=SEARCH_SPACE,
)

print("=" * 60)
print("STARTING RAY TUNE SWEEP")
print(f"  Trials     : 12")
print(f"  Max epochs : 30 per trial")
print(f"  Scheduler  : ASHA (kills bad trials after 5 epochs)")
print(f"  Search     : Optuna TPE (smarter than random)")
print("=" * 60)

sweep_start = time.time()
results     = tuner.fit()
sweep_min   = (time.time() - sweep_start) / 60

print(f"\n Sweep complete in {sweep_min:.1f} minutes")

2026-03-23 20:54:43,812	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/root/ray_results/train_tune_2026-03-23_18-54-34' in 0.0098s.
2026-03-23 20:54:43,820	INFO tune.py:1041 -- Total run time: 7209.28 seconds (7209.23 seconds for the tuning loop).



 Sweep complete in 120.2 minutes


## Results — Best Hyperparameter Configuration

In [13]:
# ── Extract best trial ────────────────────────────────────────────────────
best_result = results.get_best_result(metric="loss", mode="min")
best_config = best_result.config
best_loss   = best_result.metrics["loss"]
best_epoch  = best_result.metrics["epoch"]

print("=" * 60)
print("BEST CONFIGURATION FOUND BY OPTUNA:")
print(f"  Learning Rate   : {best_config['lr']:.6f}")
print(f"  Batch Size      : {best_config['batch_size']}")
print(f"  Attention Heads : {best_config['num_heads']}")
print(f"  FeedForward Dim : {best_config['d_ff']}")
print(f"  Dropout         : {best_config['dropout']:.3f}")
print(f"  Num Layers      : {best_config['num_layers']}")
print(f"  Best Loss       : {best_loss:.4f}  (at epoch {best_epoch})")
print("=" * 60)

# ── Show all trial results sorted by loss ────────────────────────────────
import pandas as pd
df_results = results.get_dataframe()
cols = ["loss", "epoch", "config/lr", "config/batch_size",
        "config/num_heads", "config/d_ff", "config/dropout", "config/num_layers"]
available = [c for c in cols if c in df_results.columns]
print("\nAll trials (sorted by loss):")
display(df_results[available].sort_values("loss").reset_index(drop=True))

BEST CONFIGURATION FOUND BY OPTUNA:
  Learning Rate   : 0.000371
  Batch Size      : 64
  Attention Heads : 4
  FeedForward Dim : 1024
  Dropout         : 0.065
  Num Layers      : 3
  Best Loss       : 1.2997  (at epoch 30)

All trials (sorted by loss):


,loss,epoch,config/lr,config/batch_size,config/num_heads,config/d_ff,config/dropout,config/num_layers
0,1.299662,30,0.000371,64,4,1024,0.064823,3
1,1.365827,30,0.000329,16,8,2048,0.056429,3
2,1.455317,30,0.000371,16,8,2048,0.100845,3
3,1.662927,30,0.000112,64,8,2048,0.122051,4
4,3.039064,10,0.000129,64,4,2048,0.202542,4
5,4.346699,5,0.000101,16,4,1024,0.347716,3
6,4.410960,5,0.000309,16,8,2048,0.317451,4
7,4.525947,5,0.000038,32,4,1024,0.172031,3
8,5.226710,5,0.000019,32,4,1024,0.303654,4
9,5.332004,5,0.000012,16,8,1024,0.306284,3


## Retrain Best Model from Scratch
Using the best configuration for a clean final evaluation.

In [14]:
src_pad = en_vocab["<pad>"]
tgt_pad = hi_vocab["<pad>"]

best_dataset = TranslationDataset(df, en_vocab, hi_vocab, max_len=MAX_LEN)
best_loader  = DataLoader(best_dataset, batch_size=best_config["batch_size"],
                          shuffle=True, collate_fn=collate_fn,
                          num_workers=2, pin_memory=True)

best_model = Transformer(
    src_vocab  = len(en_vocab),
    tgt_vocab  = len(hi_vocab),
    d_model    = D_MODEL,
    num_layers = best_config["num_layers"],
    num_heads  = best_config["num_heads"],
    d_ff       = best_config["d_ff"],
    max_len    = MAX_LEN,
    dropout    = best_config["dropout"],
).to(DEVICE)

optimizer = optim.Adam(best_model.parameters(), lr=best_config["lr"],
                       betas=(0.9, 0.98), eps=1e-9)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=best_config["num_epochs"], eta_min=best_config["lr"] * 0.1)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_pad, label_smoothing=0.1)

print(f"Retraining for {best_config['num_epochs']} epochs with best config...")
retrain_start = time.time()
final_loss = 0.0

for epoch in range(best_config["num_epochs"]):
    best_model.train()
    epoch_loss = 0.0
    for src, tgt_in, tgt_out in best_loader:
        src, tgt_in, tgt_out = src.to(DEVICE), tgt_in.to(DEVICE), tgt_out.to(DEVICE)
        logits = best_model(src, tgt_in, src_pad, tgt_pad)
        loss   = criterion(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(best_model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    final_loss = epoch_loss / len(best_loader)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch [{epoch+1:>2}/{best_config['num_epochs']}]  Loss: {final_loss:.4f}")

retrain_min = (time.time() - retrain_start) / 60
print(f"\nRetraining complete in {retrain_min:.1f} minutes")

Retraining for 30 epochs with best config...
  Epoch [ 1/30]  Loss: 5.1537
  Epoch [ 5/30]  Loss: 2.4968
  Epoch [10/30]  Loss: 1.6410
  Epoch [15/30]  Loss: 1.4561
  Epoch [20/30]  Loss: 1.3707
  Epoch [25/30]  Loss: 1.3202
  Epoch [30/30]  Loss: 1.3000

Retraining complete in 14.7 minutes


## Evaluate BLEU Score

In [15]:
def translate(model, sentence, en_v, hi_v, device, max_len=MAX_LEN):
    """Greedy decode — generates Hindi translation token by token."""
    model.eval()
    src = torch.tensor(encode_sentence(sentence, en_v, max_len)).unsqueeze(0).to(device)
    toks = [hi_v["<sos>"]]
    for _ in range(max_len):
        t   = torch.tensor(toks).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(src, t, en_v["<pad>"], hi_v["<pad>"])
        nxt = out[0, -1].argmax().item()
        toks.append(nxt)
        if nxt == hi_v["<eos>"]: break
    return " ".join(hi_v.itos[t] for t in toks[1:-1])


# Validation set
VAL_PAIRS = [
    ("I love you.",                  "मैं तुमसे प्यार करता हूँ।"),
    ("How are you?",                 "आप कैसे हैं?"),
    ("You should sleep.",            "आपको सोना चाहिए।"),
    ("Maybe Tom doesn't love you.",  "टॉम शायद तुमसे प्यार नहीं करता है।"),
    ("Let me tell Tom.",             "मुझे टॉम को बताने दीजिए।"),
]

smoothie = SmoothingFunction().method4
refs, hyps = [], []
for en_s, hi_s in VAL_PAIRS:
    pred = translate(best_model, en_s, en_vocab, hi_vocab, DEVICE)
    hyps.append(pred.split())
    refs.append([hi_s.split()])

bleu = corpus_bleu(refs, hyps, smoothing_function=smoothie)

# ── Final comparison table ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("FINAL RESULTS COMPARISON")
print(f"{'Metric':<28}{'Baseline':>14}{'Best Model':>14}")
print("-" * 56)
print(f"{'Epochs':<28}{BASELINE['epochs']:>14}{best_config['num_epochs']:>14}")
print(f"{'BLEU Score':<28}{'52.47%':>14}{bleu*100:>13.2f}%")
print(f"{'Final Loss':<28}{'0.0972':>14}{final_loss:>14.4f}")
print(f"{'Retrain Time (min)':<28}{'128':>14}{retrain_min:>14.1f}")
beat = "YES ✓" if bleu >= BASELINE["bleu_score"] else "NO ✗"
print(f"{'Beat Baseline?':<28}{'—':>14}{beat:>14}")
print("=" * 60)

# ── Sample translations ───────────────────────────────────────────────────
print("\nSAMPLE TRANSLATIONS:")
tests = ["I love you.", "What is your name?", "How are you?",
         "The weather is nice today.", "She is a good teacher."]
for s in tests:
    hi = translate(best_model, s, en_vocab, hi_vocab, DEVICE)
    print(f"  EN: {s}")
    print(f"  HI: {hi}\n")


FINAL RESULTS COMPARISON
Metric                            Baseline    Best Model
--------------------------------------------------------
Epochs                                 100            30
BLEU Score                          52.47%        71.47%
Final Loss                          0.0972        1.3000
Retrain Time (min)                     128          14.7
Beat Baseline?                           —         YES ✓

SAMPLE TRANSLATIONS:
  EN: I love you.
  HI: मैं आपसे प्यार करती हूँ।

  EN: What is your name?
  HI: आपका नाम क्या है?

  EN: How are you?
  HI: आप कैसे हो?

  EN: The weather is nice today.
  HI: मौसम आज का मौसम है।

  EN: She is a good teacher.
  HI: वह अच्छी अध्यापिका है.



## Save Best Model to Google Drive

In [16]:
import pickle, os

# Save model weights
model_path = f"{SAVE_DIR}/m25csa018_ass_4_best_model.pth"
torch.save(best_model.state_dict(), model_path)
print(f" Model saved → {model_path}")

# Save vocabs
with open(f"{SAVE_DIR}/en_vocab.pkl", "wb") as f: pickle.dump(en_vocab, f)
with open(f"{SAVE_DIR}/hi_vocab.pkl", "wb") as f: pickle.dump(hi_vocab, f)


 Model saved → /kaggle/working/m25csa018_ass_4_best_model.pth
